# Graphics Complexity Evaluator

**The Graphics Complexity Evaluator** assesses how demanding a text's graphics are for students at a given grade level. It looks at each chart, map, diagram, table, photograph, or illustration that accompanies a passage — how dense it is, how many layers of information it carries, how much interpretation it asks for, how abstract its conventions are — and it checks whether any of the graphics have to be read against each other to be usable at all. It then weighs those demands against what students at the target grade are expected to handle. When you run a passage and its graphics through the evaluator, it returns a structured output with the following top-level fields:

* **complexity_score**: The Graphics Complexity level (Slightly to Exceedingly Complex), or `more_context_needed` when the excerpt gives no graphic to rate.
* **reasoning**: A synthesis of why the text's graphics fit the chosen complexity level.
* **details**: Keyed instructional detail, containing:
    * **detailed_summary**: Individual graphics complexity factors that drive the rating, with descriptions and their effect on the dimension.
    * **adjustment_and_scaffolding**: Scaffolding strategies to make the text's graphics accessible at the target grade.
    * **recommended_use_cases**: Additional instructional opportunities for using the text's graphics.

This evaluator is multimodal: alongside the passage text it takes one image per graphic in scope, attached to the prompt as image blocks rather than substituted into it as text. To reach the score above, the model first rates each graphic on its own and checks whether any must be read jointly; those per-graphic working fields are validated inside this notebook and then dropped, so the returned output carries the same three fields as every other evaluator in this repository.

This gives you a clear signal about the graphical demands of a passage, helping ensure AI-generated content is appropriate for the target grade.

In [ ]:
%pip install -qU langchain-google-genai langchain pydantic textstat typing_extensions python-dotenv


In [ ]:
import base64
import copy
import getpass
import hashlib
import io
import json
import os
import pprint as pp
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse
from urllib.request import Request, urlopen

import textstat
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Check for the API key
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [ ]:
ASSETS_DIR = Path(".")
config_path = ASSETS_DIR / "config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

# Load standalone schema files (config.json references them via $ref by path).
with open(ASSETS_DIR / "input_schema.json") as f:
    INPUT_SCHEMA = json.load(f)
with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

# Load every prompt message declared in config (system, user, ...). Each
# message has {role, source_path, sha256}. We verify each file's sha256
# matches the declared hash -- drift tripwire #1, applied to every prompt
# regardless of role. CI should promote a mismatch to a hard failure.
PROMPT_MESSAGES = []  # list of (role, text) tuples, preserving config order
for msg_spec in CONFIG["steps"][0]["prompt"]["messages"]:
    role = msg_spec["role"]
    path = ASSETS_DIR / msg_spec["source_path"]
    raw = path.read_bytes()
    text = raw.decode("utf-8")
    actual_sha = hashlib.sha256(raw).hexdigest()
    declared_sha = msg_spec["sha256"]
    assert actual_sha == declared_sha, (
        f"prompt drift detected for role={role!r} ({msg_spec['source_path']}): "
        f"declared {declared_sha[:12]}..., actual on disk {actual_sha[:12]}..."
    )
    PROMPT_MESSAGES.append((role, text))

# Rubric vocabulary, read from output_schema.json rather than hardcoded.
# more_context_needed is a flag meaning "no graphic to rate", NOT a fifth rung:
# it is excluded from RUBRIC_ORDER so that no distance or adjacency comparison
# can ever treat it as one step away from exceedingly_complex.
SCORE_FIELD = CONFIG["outcome"]["score"]
ALL_SCORES = OUTPUT_SCHEMA["properties"][SCORE_FIELD]["enum"]
NEEDS_CONTEXT = "more_context_needed"
RUBRIC_ORDER = [s for s in ALL_SCORES if s != NEEDS_CONTEXT]
RANK = {s: i for i, s in enumerate(RUBRIC_ORDER)}

print(
    f"Loaded {CONFIG['evaluator']['id']} "
    f"from {ASSETS_DIR.resolve()}"
)
print(f"  model:       {CONFIG['steps'][0]['model']['name']}")
print(f"  temperature: {CONFIG['steps'][0]['generation']['temperature']}")
print(f"  rubric:      {RUBRIC_ORDER} + flag {[NEEDS_CONTEXT]}")
print(f"  prompts:")
for msg_spec, (role, text) in zip(CONFIG["steps"][0]["prompt"]["messages"], PROMPT_MESSAGES):
    sha = hashlib.sha256((ASSETS_DIR / msg_spec["source_path"]).read_bytes()).hexdigest()[:12]
    print(f"    {role:>6}  {msg_spec['source_path']:<14} ({len(text):>5} chars, sha {sha})")

In [ ]:
# -------------------------------------------------------------------------
# FK score helper (declared as a preprocessing step in CONFIG['preprocessing'])
# -------------------------------------------------------------------------
_FK_PRE = next(p for p in CONFIG["preprocessing"] if p["id"] == "fk_score")
_FK_IMPL = _FK_PRE["implementation"]["python"]
_FK_LIB = _FK_IMPL["library"]
_FK_FN = _FK_IMPL["function"]
_FK_TRANSFORM = _FK_IMPL["post_transform"]

if _FK_LIB != "textstat":
    raise ValueError(f"unsupported fk library in config: {_FK_LIB!r}")


def calculate_fk_score(text) -> float:
    """Compute Flesch-Kincaid Grade Level per CONFIG['preprocessing']."""
    fn = getattr(textstat, _FK_FN)
    value = fn(text)
    if _FK_TRANSFORM["type"] == "round":
        value = round(value, _FK_TRANSFORM["precision"])
    else:
        raise ValueError(f"unsupported post_transform type: {_FK_TRANSFORM['type']!r}")
    return value


# -------------------------------------------------------------------------
# Image helper (declared as the graphics_blocks preprocessing step in config)
# -------------------------------------------------------------------------
# Each entry of input.graphics is a path relative to this directory or an
# http(s) URL. The media type is read from the file's first bytes rather than
# its extension, because a screenshot saved as .png is not always a PNG. The
# bytes are passed through untouched -- whatever the model sees here is
# byte-identical to the file on disk.
_SIGNATURES = [
    (lambda b: b.startswith(b"\x89PNG\r\n\x1a\n"),              "image/png"),
    (lambda b: b.startswith(b"\xff\xd8\xff"),                     "image/jpeg"),
    (lambda b: b[:6] in (b"GIF87a", b"GIF89a"),                    "image/gif"),
    (lambda b: b[:4] == b"RIFF" and b[8:12] == b"WEBP",            "image/webp"),
]


def read_image_bytes(reference: str) -> bytes:
    """Read one graphic from a local path (relative to ASSETS_DIR) or an http(s) URL."""
    parsed = urlparse(reference)
    if parsed.scheme.lower() in ("http", "https"):
        request = Request(reference, headers={"User-Agent": "graphics-complexity-evaluator"})
        with urlopen(request, timeout=30) as response:
            return response.read()
    if parsed.scheme:
        raise ValueError(f"unsupported image URL scheme: {parsed.scheme!r}")
    path = Path(reference).expanduser()
    return (path if path.is_absolute() else ASSETS_DIR / path).read_bytes()


def to_data_url(graphics):
    """Implements CONFIG['preprocessing'] id=graphics_blocks: ordered image blocks."""
    blocks = []
    for reference in graphics:
        data = read_image_bytes(reference)
        mime = next((m for test, m in _SIGNATURES if test(data)), None)
        if mime is None:
            raise ValueError(f"unsupported image signature for {reference!r}; expected PNG, JPEG, GIF or WEBP")
        encoded = base64.b64encode(data).decode("ascii")
        blocks.append({"type": "image_url", "image_url": {"url": f"data:{mime};base64,{encoded}"}})
    return blocks


def resolve_figure_labels(graphics, figure_labels=None):
    """{figure_labels} is optional; default to Image 1..N so the Nth label is the Nth image."""
    if not (figure_labels or "").strip():
        return [f"Image {i}" for i in range(1, len(graphics) + 1)]
    labels = [label.strip() for label in figure_labels.split(",")]
    if len(labels) != len(graphics) or not all(labels) or len(set(labels)) != len(labels):
        raise ValueError("supply one distinct, non-empty figure label per graphic, in order")
    return labels


# -------------------------------------------------------------------------
# Internal output schema: what the MODEL is asked to return
# -------------------------------------------------------------------------
# output_schema.json is the public contract -- the three fields a caller gets
# back. The model is asked for three more on top of it: a per-graphic rating,
# a joint-reading judgement, and which of the two produced the final score.
# Those are the model's working-out. We ask for them because the final score
# is defined in terms of them, and because they let this notebook check that
# the score the model reported actually follows from the ratings it gave.
# They are validated below and then dropped; they are not returned.
def _strict(properties, **extra):
    return {"type": "object", "properties": properties,
            "required": list(properties), "additionalProperties": False, **extra}


INTERNAL_OUTPUT_SCHEMA = copy.deepcopy(OUTPUT_SCHEMA)
for key in ("$schema", "$id"):
    INTERNAL_OUTPUT_SCHEMA.pop(key, None)
INTERNAL_OUTPUT_SCHEMA["title"] = "GraphicsEvaluatorInternalOutput"
INTERNAL_OUTPUT_SCHEMA.setdefault("$defs", {}).update({
    "GraphicAssessment": _strict({
        "label": {"type": "string",
                  "description": "The supplied figure label identifying this graphic."},
        "role": {"type": "string",
                 "enum": ["decorative", "supporting", "extending", "complicating"],
                 "description": "This graphic's relationship to the surrounding text."},
        "demand_notes": {"type": "string",
                         "description": "What drives this graphic's demand: density, layering, "
                                        "interpretive elements, abstraction."},
        "complexity": {"type": "string", "enum": ALL_SCORES,
                       "description": "This graphic's own rating."},
    }),
    "JointReading": _strict({
        "applies": {"type": "boolean",
                    "description": "True only if the reader must hold two or more graphics "
                                   "together to use either. Several graphics on a page is not "
                                   "by itself a joint reading."},
        "which": {"type": "array", "items": {"type": "string"},
                  "description": "The labels read together (at least two) when applies is true; "
                                 "otherwise empty."},
        "complexity": {"type": ["string", "null"], "enum": [*RUBRIC_ORDER, None],
                       "description": "The grouping rated as one thing. Null when applies is false."},
        "reasoning": {"type": "string",
                      "description": "What the reader has to carry from one graphic to the other."},
    }),
})
INTERNAL_OUTPUT_SCHEMA["properties"].update({
    "graphics": {"type": "array", "items": {"$ref": "#/$defs/GraphicAssessment"},
                 "description": "Exactly one entry per attached image, in the supplied order."},
    "joint_reading": {"$ref": "#/$defs/JointReading"},
    "aggregation_rule": {"type": "string", "enum": ["highest_individual", "joint_reading"],
                         "description": "Which rule produced complexity_score."},
})
INTERNAL_OUTPUT_SCHEMA["required"] = list(INTERNAL_OUTPUT_SCHEMA["properties"])


# -------------------------------------------------------------------------
# Output contract: does the reported score follow from the model's own ratings?
# -------------------------------------------------------------------------
# The provider guarantees the SHAPE of the response; these checks cover the
# things a schema cannot express. A violation is an error, not a low score:
# an output that contradicts itself is not evidence about the passage.
class OutputContractError(ValueError):
    def __init__(self, violations, internal_output=None):
        self.violations = violations
        self.internal_output = internal_output
        super().__init__("; ".join(violations))


def check_output_contract(output, n_images):
    """Raise if the model's per-graphic working-out does not support its final score."""
    violations = []
    graphics = output["graphics"]
    labels = [g["label"] for g in graphics]

    # Match system.txt: each image is one graphic, even when it contains
    # multiple panels. Require exactly one assessment per attached image.
    # Distinct labels let joint_reading refer unambiguously to those graphics.
    if len(graphics) != n_images:
        violations.append(
            f"graphics has {len(graphics)} entries for {n_images} image(s) -- exactly one entry per image is required")
    if len(set(labels)) != len(labels):
        violations.append(f"graphics labels must be distinct so joint_reading can refer to them: {labels}")

    joint = output["joint_reading"]
    if joint["applies"]:
        if len(set(joint["which"])) < 2:
            violations.append("joint_reading.applies is true but `which` names fewer than two graphics")
        if not set(joint["which"]).issubset(labels):
            violations.append("joint_reading.which names a graphic that is not in graphics[]")
        if joint["complexity"] not in RANK:
            violations.append("joint_reading.applies is true but its complexity is missing or off-scale")
    elif joint["which"] or joint["complexity"] is not None:
        violations.append("joint_reading.applies is false, so `which` must be empty and complexity null")

    # The excerpt's rating is the highest of the per-graphic ratings and the
    # joint rating where one applies. more_context_needed is off-scale: the
    # final score is the flag only when NO graphic could be rated at all.
    individual = [g["complexity"] for g in graphics]
    candidates = individual + ([joint["complexity"]] if joint["applies"] else [])
    ordinal = [c for c in candidates if c in RANK]
    implied = max(ordinal, key=RANK.__getitem__) if ordinal else NEEDS_CONTEXT
    if output[SCORE_FIELD] != implied:
        violations.append(
            f"{SCORE_FIELD}={output[SCORE_FIELD]} but the per-graphic and joint ratings imply {implied}")

    if output["aggregation_rule"] == "joint_reading":
        if not joint["applies"] or joint["complexity"] != implied:
            violations.append("aggregation_rule=joint_reading but no active joint rating set the score")
    elif implied in RANK and implied not in individual:
        violations.append("aggregation_rule=highest_individual but no individual rating set the score")

    if violations:
        raise OutputContractError(violations, output)


# -------------------------------------------------------------------------
# Evaluator function: model / prompt / parser config all read from CONFIG
# -------------------------------------------------------------------------
_STEP = CONFIG["steps"][0]  # single-step evaluator today. Extensible to multi-step evaluators.


def build_structured_model():
    """One model handle, reused across calls so a fixture run does not rebuild it each time."""
    llm = ChatGoogleGenerativeAI(
        model=_STEP["model"]["name"],
        temperature=_STEP["generation"]["temperature"],
    )
    # method="json_schema" asks Gemini to constrain decoding to the schema
    # directly, rather than routing structured output through tool-calling --
    # the more reliable path for a nested schema like the internal one.
    return llm.with_structured_output(INTERNAL_OUTPUT_SCHEMA, method="json_schema", include_raw=True)


def evaluate_text_complexity(text: str, grade_level: str, graphics, figure_labels: str = None,
                             structured_model=None):
    """
    Evaluate the Graphics Complexity-dimension complexity of a text and its graphics
    using the canonical config in this directory (config.json + system.txt + user.txt).

    `graphics` is a list of image paths (relative to this directory) or http(s) URLs,
    one per graphic in scope. `figure_labels` is an optional comma-separated list of
    labels in the same order; it defaults to "Image 1, Image 2, ...".

    Returns a dict with full I/O trace fields:
      - rendered_prompt:  the actual list of messages sent to the model
                          (input-side trace, image blocks included).
      - raw_output:       the AIMessage object returned by the LLM
                          (preserves response_metadata, usage_metadata).
      - raw_text:         just the string content of the AIMessage.
      - formatted_output: the parsed dict matching OUTPUT_SCHEMA -- three fields.
      - internal_output:  the model's six-field working-out, kept for QA only.
      - usage:            token-usage metadata if the provider returned it.

    The LLM is invoked ONCE; include_raw=True returns both the raw AIMessage
    and the parsed output without a second call.
    """
    model = structured_model if structured_model is not None else build_structured_model()
    prompt_template = ChatPromptTemplate.from_messages(PROMPT_MESSAGES)

    try:
        # Step A: Preprocessing -- FK score from the text, image blocks from the graphics.
        labels = resolve_figure_labels(graphics, figure_labels)
        fk_score = calculate_fk_score(text)
        image_blocks = to_data_url(graphics)
        print(f"Calculated Flesch-Kincaid Score: {fk_score}")

        inputs = {"text": text, "grade_level": grade_level, "fk_score": fk_score,
                  "figure_labels": ", ".join(labels)}

        # Step B: Render the prompt up-front so we can return exactly what was
        #         sent to the model, then attach the images to the user message.
        #         The graphics are content blocks appended after the rendered
        #         text -- they are never substituted into the prompt string.
        rendered_messages = prompt_template.format_messages(**inputs)
        rendered_messages[-1] = HumanMessage(
            content=[{"type": "text", "text": rendered_messages[-1].content}, *image_blocks])

        # Step C: Single LLM call -> raw AIMessage + parsed output dict.
        #         No second LLM call.
        raw = model.invoke(rendered_messages)

        if raw.get("parsing_error"):
            raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

        # Step D: Check the model's working-out, then drop it. Only the fields
        #         declared in output_schema.json are returned to the caller.
        internal = raw["parsed"]
        check_output_contract(internal, len(graphics))
        public = {k: copy.deepcopy(internal[k]) for k in OUTPUT_SCHEMA["properties"]}

        # Step E: Return the full trace dict.
        return {
            "rendered_prompt": [m.model_dump() for m in rendered_messages],
            "raw_output":       raw["raw"],
            "raw_text":         raw["raw"].content,
            "formatted_output": public,
            "internal_output":  internal,
            "usage":            getattr(raw["raw"], "usage_metadata", None),
        }
    except Exception as e:
        return f"Error evaluating text: {e}"

In [ ]:
# One passage and its graphics. The sample is taken from fixtures.json so the
# text and the images belong together; the call itself shows the full signature.
with open(ASSETS_DIR / CONFIG["fixtures"]["path"]) as f:
    _fixtures = json.load(f)
sample = next(fx for fx in _fixtures if fx["id"] == "SL-019_intro_methods")

result = evaluate_text_complexity(
    text=sample["input"]["text"],
    grade_level=sample["input"]["grade_level"],
    graphics=sample["input"]["graphics"],          # ["images/....png", "images/....png"]
    figure_labels=sample["input"]["figure_labels"],
)
pp.pprint(result["formatted_output"] if isinstance(result, dict) else result)

In [ ]:
# I/O trace breakdown -- this is what the SDK engineer will replicate in TS.
# The rendered prompt is printed with the base64 image payloads truncated, so
# the structure of the multimodal message stays readable.
def _shorten(messages):
    trimmed = copy.deepcopy(messages)
    for message in trimmed:
        content = message.get("content")
        if isinstance(content, list):
            for block in content:
                url = block.get("image_url", {}).get("url", "")
                if url.startswith("data:"):
                    head = url.split(",", 1)[0]
                    block["image_url"]["url"] = f"{head},<{len(url)} chars of base64>"
    return trimmed


if not isinstance(result, dict):
    raise SystemExit(result)

print("=" * 60)
print("RENDERED PROMPT (input sent to the LLM)")
print("=" * 60)
pp.pprint(_shorten(result["rendered_prompt"]))

print("\n" + "=" * 60)
print("RAW LLM TEXT (model's verbatim output)")
print("=" * 60)
print(result["raw_text"])

print("\n" + "=" * 60)
print("PARSED OUTPUT (output_schema)")
print("=" * 60)
pp.pprint(result["formatted_output"])

print("\n" + "=" * 60)
print("INTERNAL WORKING-OUT (QA only -- not returned to callers)")
print("=" * 60)
pp.pprint({k: result["internal_output"][k] for k in ("graphics", "joint_reading", "aggregation_rule")}
         )

print("\n" + "=" * 60)
print("USAGE METADATA")
print("=" * 60)
pp.pprint(result["usage"])

In [ ]:
# -------------------------------------------------------------------------
# Sniff-test runner: load fixtures.json and check predictions against expected
# -------------------------------------------------------------------------
# Fixtures live next to config.json + system.txt + user.txt, with their images
# in images/. Each case has:
#   - id, description, source (optional)
#   - input: {text, grade_level, graphics, figure_labels}  -- runtime evaluator inputs
#   - expected: {complexity_score}                         -- ground-truth label from rubric
#
# We test that single value only -- the model's free-text 'reasoning' field is
# non-deterministic across runs. Note that the score itself moves between
# adjacent levels from run to run as well, which is what the tolerance below
# is for: these fixtures are a drift tripwire, not an accuracy measurement.

fixtures_path = ASSETS_DIR / CONFIG["fixtures"]["path"]
with open(fixtures_path) as f:
    fixtures = json.load(f)
print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name} "
      f"({sum(len(fx['input']['graphics']) for fx in fixtures)} images)\n")

# Adjacency tolerance per CONFIG['fixtures']['tolerance']. RUBRIC_ORDER excludes
# more_context_needed, so the flag can never earn adjacent credit against a level.
_ALLOW_ADJ = bool(CONFIG["fixtures"]["tolerance"].get("allow_adjacent_levels", False))


def _score_outcome(predicted: str, expected: str):
    """Return ('exact' | 'adjacent' | 'fail', distance_or_None)."""
    if predicted == expected:
        return "exact", 0
    if _ALLOW_ADJ and predicted in RANK and expected in RANK:
        d = abs(RANK[predicted] - RANK[expected])
        if d == 1:
            return "adjacent", d
    return "fail", None


# Run each fixture, accumulate results. One model handle for the whole run.
_model = build_structured_model()
results = []
for fx in fixtures:
    expected = fx["expected"][SCORE_FIELD]
    out = evaluate_text_complexity(
        text=fx["input"]["text"],
        grade_level=fx["input"]["grade_level"],
        graphics=fx["input"]["graphics"],
        figure_labels=fx["input"].get("figure_labels"),
        structured_model=_model,
    )
    if isinstance(out, str):  # error path -- stays in the denominator
        results.append({"id": fx["id"], "status": "error", "predicted": None,
                        "expected": expected, "error": out})
        continue
    predicted = out["formatted_output"][SCORE_FIELD]
    status, _ = _score_outcome(predicted, expected)
    results.append({
        "id": fx["id"], "status": status,
        "predicted": predicted, "expected": expected,
        "description": fx.get("description", ""),
    })

# Per-case report
print("\n" + "=" * 92)
print(f"{'ID':>22}  {'STATUS':<8}  {'PREDICTED':<22}  {'EXPECTED':<22}  DESCRIPTION")
print("=" * 92)
for r in results:
    icon = {"exact": "PASS", "adjacent": "PASS*", "fail": "FAIL", "error": "ERR"}[r["status"]]
    print(f"{r['id']:>22}  {icon:<8}  {(r['predicted'] or '-'):<22}  {r['expected']:<22}  {r.get('description','')[:25]}")
    if r["status"] == "error":
        print(f"{'':>22}  {r['error'][:100]}")

# Summary
n_total = len(results)
n_exact = sum(1 for r in results if r["status"] == "exact")
n_adj   = sum(1 for r in results if r["status"] == "adjacent")
n_fail  = sum(1 for r in results if r["status"] == "fail")
n_err   = sum(1 for r in results if r["status"] == "error")
print("=" * 92)
print(f"Summary: {n_exact} exact, {n_adj} adjacent (tolerated), {n_fail} fail, {n_err} error  --  total {n_total}")
if _ALLOW_ADJ:
    print("(Adjacency tolerance ON: predictions within +/-1 rubric step of the expected label count as PASS*.)")